# Two Gaussians: entropy of MGD vs the exact max-ent

Gaussian signals with the Lagrangian (turbulence) and Eulerian (jets) data spectrum, model = 9 Scat2 Q1 band energies, d = 256.
Everything is exact for a Gaussian: $H(p^*)$ (max-ent with the 9 moments), $H$(data), $H(p_1^\sigma)$ of the MGD samples.
Paper: $H_\sigma^* \le H(p_1^\sigma) \le H(p^*)$. Entropies in nats. Run from the repo root on JZ.

In [ ]:
%matplotlib inline
import sys, json, csv
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / 'codes').is_dir() else Path.cwd().parent
sys.path[:0] = [str(ROOT), str(ROOT / 'codes'), str(ROOT / 'turbulence'), str(ROOT / 'gauss_spectrum')]
import logz_gauss_benchmark as B, logp_gauss_scat2 as lg
from sigma_sweep_summary import load, col
OUT = ROOT / 'gauss_spectrum' / 'sigma_sweep'
NAMES = {'turbulence': 'Lagrangian', 'jets': 'Eulerian'}
data = load(OUT)
{ds: [r['sigma'] for r in runs] for ds, runs in data.items()}

## 1. Paper Fig. 2(b): entropies relative to $H(p^*)$ vs $\sigma^2$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (ds, runs) in zip(axes, data.items()):
    s2 = np.array([r['sigma'] ** 2 for r in runs])
    m = lambda k: np.array([np.nanmean(col(r['rows'], k)) for r in runs])
    Hs = m('H_star')
    ax.axhline(0, color='C3', lw=2, label=r'$H(p^*)$ max-ent')
    ax.axhline(np.mean(m('H_exact_1') - Hs), color='0.5', ls='--', label='data Gaussian')
    ax.plot(s2, m('H_gen_1') - Hs, 'o-', color='C0', label=r'$H(p_1^\sigma)$ samples')
    ax.plot(s2, m('H_bound_1') - Hs, 'o-', color='k', label=r'$H_\sigma^*$ bound')
    ax.set_xscale('log'); ax.set_yscale('symlog', linthresh=10); ax.minorticks_off()
    ax.set_xticks(s2, [f"{a:.2g}\n{r['nt']}" for a, r in zip(s2, runs)], fontsize=7)
    ax.set_xlabel(r'$\sigma^2$ (second line: nt)'); ax.set_ylabel(r'entropy $- H(p^*)$ (nats)')
    ax.set_title(NAMES[ds]); ax.legend(fontsize=8, frameon=False)
fig.tight_layout()

## 2. The two Gaussians: data spectrum, max-ent $p^*$ spectrum, MGD samples (smallest and largest $\sigma$)

In [ ]:
def first_seed(ds, sigma):
    for g in sorted((OUT / ds / 'experiments').iterdir()):
        for d in sorted(g.iterdir()) if g.is_dir() else []:
            if (d / 'config.json').exists():
                cfg = json.loads((d / 'config.json').read_text()); cfg = cfg.get('args', cfg)
                if abs(float(cfg['sigma']) - sigma) < 1e-9:
                    return d, cfg
    raise FileNotFoundError(f'{ds} sigma {sigma}')

def spectra(ds, run):
    seed_dir, cfg = first_seed(ds, run['sigma'])
    M = int(cfg['M'])
    pots = lg.build_pots(cfg, 'cpu'); psi2 = lg.filters_of(pots)
    x = lg.load_data_x1(cfg, 'cpu', seed_dir)
    star = B.max_ent_gaussian(x, psi2, M)
    xt = lg.load_samples(OUT / ds, seed_dir.name).reshape(-1, M).double().numpy()
    gen = (np.abs(np.fft.fft(xt, axis=-1, norm='ortho')) ** 2).mean(0)
    return M, psi2, star, gen

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (ds, runs) in zip(axes, data.items()):
    for run, c in ((runs[0], 'C0'), (runs[-1], 'C2')):
        M, psi2, star, gen = spectra(ds, run)
        w = np.arange(1, M // 2 + 1)
        ax.loglog(w, gen[w], color=c, lw=1, label=rf"MGD samples $\sigma$={run['sigma']:g} (nt {run['nt']})")
    ax.loglog(w, star['spectrum_data'][w], 'k', lw=2, label='data')
    ax.loglog(w, star['spectrum_star'][w], 'C3--', lw=2, label=r'max-ent $p^*$')
    ax2 = ax.twinx(); ax2.plot(w, psi2[:, w].T, color='0.8', lw=0.8); ax2.set_yticks([]); ax2.set_xscale('log')
    ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
    ax.set_xlabel('frequency index'); ax.set_ylabel('power spectrum'); ax.set_title(f'{NAMES[ds]} (grey: the 9 filters)')
    ax.legend(fontsize=8, frameon=False)
    print(f"{NAMES[ds]}: H(p*) - H(data) = KL(data||p*) = {star['H_star'] - 0.5 * M * np.log(2 * np.pi * np.e) - 0.5 * np.log(star['spectrum_data']).sum():+.2f} nats")
fig.tight_layout()